In [ ]:
"""
================================================================================
  SOIL MOISTURE 10-DAY AHEAD PREDICTION — TRANSFORMER MODEL (OPTIMIZED & FIXED)
================================================================================
"""
# ─────────────────────────────────────────────────────────────────────────────
# 0.  IMPORTS
# ─────────────────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy.interpolate import interp1d
import matplotlib.pyplot as plt
import warnings
import os

warnings.filterwarnings("ignore")
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {device}")

# ─────────────────────────────────────────────────────────────────────────────
# 1.  CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────
CFG = dict(
    # ── Data ──────────────────────────────────────────────────────────────
    DATA_PATH    = "/content/Final_Dataset.xlsx",   # ← change path if needed
    RESAMPLE_MIN = 10,     # resample raw 1-min data to 10-min blocks
    # ── Sequence (in 10-min blocks after resampling) ───────────────────────
    SEQ_LEN      = 144,    # look-back: 144 × 10min = 1,440 min = 1 day
    HORIZON      = 1440,   # predict:  1440 × 10min = 14,400 min = 10 days
    # ── Split ─────────────────────────────────────────────────────────────
    TRAIN_FRAC   = 0.70,
    VAL_FRAC     = 0.15,
    # ── Model ─────────────────────────────────────────────────────────────
    D_MODEL      = 128,    # transformer embedding dimension
    N_HEADS      = 4,      # attention heads  (128/4 = 32 per head)
    N_LAYERS     = 4,      # encoder blocks
    D_FF         = 512,    # feed-forward hidden size
    DROPOUT      = 0.10,
    # ── Training ──────────────────────────────────────────────────────────
    BATCH_SIZE   = 64,
    LR           = 3e-4,
    WEIGHT_DECAY = 0.0,    # Set to 0 to prevent flattening/forcing to mean
    EPOCHS       = 100,    # Expanded epochs
    PATIENCE     = 35,     # Increased patience for deeper landscape exploration
    # ── Output ────────────────────────────────────────────────────────────
    SAVE_MODEL   = "best_sm_minute_transformer.pt",
    FIGURE_DIR   = ".",
)

# 10 features including sinusoidal time signatures
FEATURE_COLS = [
    "irrigation",    # irrigation rate (10-min avg, forward-filled)
    "sm",            # soil moisture (10-min avg)
    "sm_roll6",      # 1-hour  rolling mean  (6 × 10min = 60min)
    "sm_roll24",     # 4-hour  rolling mean  (24 × 10min = 240min)
    "sm_diff1",      # 10-min rate of change
    "sm_lag1",       # lag-1  (10 min ago)
    "sm_lag6",       # lag-6  (1 hour ago)
    "sm_lag24",      # lag-24 (4 hours ago)
    "sin_time",      # Cyclical diurnal feature (Sine component)
    "cos_time"       # Cyclical diurnal feature (Cosine component)
]
N_FEATURES = len(FEATURE_COLS)   # 10

# ─────────────────────────────────────────────────────────────────────────────
# 2.  LOAD & EXPLORE
# ─────────────────────────────────────────────────────────────────────────────
def load_and_explore(path: str) -> pd.DataFrame:
    print("\n" + "=" * 65)
    print("STEP 1 — Load & Explore Raw Data")
    print("=" * 65)
    df = pd.read_excel(path)

    # FIX: Explicitly handle the 3-column structural layout mismatch
    print(f"  Detected columns in file: {df.columns.tolist()}")
    if len(df.columns) == 3:
        print("  Found 3 columns. Dropping the first column (assuming index/minute marker).")
        df = df.iloc[:, 1:]  # Drops the first column safely before assigning name mappings

    df.columns = ["irrigation", "sm"]
    print(f"  Raw rows (minutes) : {len(df):,}")
    print(f"  Total days         : {len(df)/1440:.2f}")
    print(f"  Irrigation NaN     : {df['irrigation'].isna().sum():,}")
    print(f"  SM NaN             : {df['sm'].isna().sum()}")
    df["irrigation"] = df["irrigation"].ffill()
    print(f"  After ffill — NaN  : {df.isnull().sum().sum()}")
    print(f"\n  Irrigation stats:")
    print(f"    min={df['irrigation'].min():.3e}  max={df['irrigation'].max():.3e}  unique={df['irrigation'].nunique()}")
    print(f"\n  Soil Moisture stats:")
    print(f"    min={df['sm'].min():.6f}  max={df['sm'].max():.6f}  mean={df['sm'].mean():.6f}  std={df['sm'].std():.8f}")
    return df

# ─────────────────────────────────────────────────────────────────────────────
# 3.  RESAMPLE TO 10-MINUTE INTERVALS
# ─────────────────────────────────────────────────────────────────────────────
def resample_to_10min(df: pd.DataFrame, block: int = 10) -> pd.DataFrame:
    print("\n" + "=" * 65)
    print("STEP 2 — Resample to 10-Minute Blocks")
    print("=" * 65)
    n_full = len(df) // block
    sm_arr  = df["sm"].values[:n_full * block].reshape(n_full, block).mean(axis=1)
    irr_arr = df["irrigation"].values[:n_full * block].reshape(n_full, block).mean(axis=1)
    df10 = pd.DataFrame({"irrigation": irr_arr, "sm": sm_arr})
    print(f"  Raw minutes        : {len(df):,}")
    print(f"  10-min blocks      : {len(df10):,}")
    print(f"  Total days covered : {len(df10) / 144:.2f}")
    print(f"  SM range           : {df10['sm'].min():.6f} to {df10['sm'].max():.6f}")
    return df10

# ─────────────────────────────────────────────────────────────────────────────
# 4.  FEATURE ENGINEERING
# ─────────────────────────────────────────────────────────────────────────────
def engineer_features(df10: pd.DataFrame) -> pd.DataFrame:
    print("\n" + "=" * 65)
    print("STEP 3 — Feature Engineering")
    print("=" * 65)
    df = df10.copy()
    df["sm_roll6"]  = df["sm"].rolling(6,  min_periods=1).mean()
    df["sm_roll24"] = df["sm"].rolling(24, min_periods=1).mean()
    df["sm_diff1"]  = df["sm"].diff().fillna(0)
    df["sm_lag1"]   = df["sm"].shift(1).bfill()
    df["sm_lag6"]   = df["sm"].shift(6).bfill()
    df["sm_lag24"]  = df["sm"].shift(24).bfill()

    # ── Cyclical Time Features Addition ──
    df["block_of_day"] = df.index % 144
    df["sin_time"] = np.sin(2 * np.pi * df["block_of_day"] / 144)
    df["cos_time"] = np.cos(2 * np.pi * df["block_of_day"] / 144)
    df.drop(columns=["block_of_day"], inplace=True)

    df = df.dropna().reset_index(drop=True)
    print(f"  Features ({N_FEATURES}): {FEATURE_COLS}")
    print(f"  Rows after NaN drop : {len(df):,}")
    return df

# ─────────────────────────────────────────────────────────────────────────────
# 5.  SLIDING WINDOW SEQUENCES
# ─────────────────────────────────────────────────────────────────────────────
def make_windows(df: pd.DataFrame, seq_len: int, horizon: int) -> tuple:
    print("\n" + "=" * 65)
    print("STEP 4 — Sliding Window Sequences")
    print("=" * 65)
    feat   = df[FEATURE_COLS].values.astype(np.float32)   # (T, 10)
    target = df["sm"].values.astype(np.float32)            # (T,)
    T      = len(feat)
    X_list, y_list = [], []
    for i in range(T - seq_len - horizon + 1):
        X_list.append(feat[i : i + seq_len])
        y_list.append(target[i + seq_len : i + seq_len + horizon])
    X = np.array(X_list, dtype=np.float32)   # (N, 144, 10)
    y = np.array(y_list, dtype=np.float32)   # (N, 1440)
    idx = np.random.permutation(len(X))
    X, y = X[idx], y[idx]
    print(f"  10-min blocks      : {T:,}")
    print(f"  Total windows      : {len(X):,}")
    print(f"  X shape            : {X.shape}  → (N, seq={seq_len}, feat={N_FEATURES})")
    print(f"  y shape            : {y.shape}  → (N, horizon={horizon})")
    print(f"  Each y = 10 days of 10-min SM = {horizon}×10 = {horizon*10:,} minutes")
    return X, y

# ─────────────────────────────────────────────────────────────────────────────
# 6.  CHRONOLOGICAL SPLIT + SCALING
# ─────────────────────────────────────────────────────────────────────────────
def split_and_scale(X, y, train_frac, val_frac):
    print("\n" + "=" * 65)
    print("STEP 5 — Train / Val / Test Split + Scaling")
    print("=" * 65)
    N     = len(X)
    t_end = int(train_frac * N)
    v_end = int((train_frac + val_frac) * N)
    X_tr, y_tr = X[:t_end],      y[:t_end]
    X_v,  y_v  = X[t_end:v_end], y[t_end:v_end]
    X_te, y_te = X[v_end:],      y[v_end:]
    print(f"  Train : {len(X_tr):>5,}  ({train_frac*100:.0f}%)")
    print(f"  Val   : {len(X_v):>5,}  ({val_frac*100:.0f}%)")
    print(f"  Test  : {len(X_te):>5,}  ({(1-train_frac-val_frac)*100:.0f}%)")
    sx = StandardScaler()
    sy = StandardScaler()
    def sc_X(arr, fit=False):
        s = arr.shape
        flat = arr.reshape(-1, N_FEATURES)
        return (sx.fit_transform(flat) if fit else sx.transform(flat)).reshape(s)
    X_tr_s = sc_X(X_tr, fit=True)
    X_v_s  = sc_X(X_v)
    X_te_s = sc_X(X_te)
    y_tr_s = sy.fit_transform(y_tr)
    y_v_s  = sy.transform(y_v)
    y_te_s = sy.transform(y_te)
    print(f"\n  Scaler fit on train only — no leakage")
    print(f"  SM target mean  : {sy.mean_[0]:.6f}")
    print(f"  SM target std   : {sy.scale_[0]:.6f}")
    return X_tr_s, y_tr_s, X_v_s, y_v_s, X_te_s, y_te, y_te_s, sx, sy

# ─────────────────────────────────────────────────────────────────────────────
# 7.  DATALOADERS
# ─────────────────────────────────────────────────────────────────────────────
def make_loaders(X_tr, y_tr, X_v, y_v, X_te, y_te_s, batch_size):
    def td(X, y):
        return TensorDataset(torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.float32))
    tl  = DataLoader(td(X_tr, y_tr),   batch_size, shuffle=True,  drop_last=False)
    vl  = DataLoader(td(X_v,  y_v),    batch_size, shuffle=False, drop_last=False)
    tel = DataLoader(td(X_te, y_te_s), batch_size, shuffle=False, drop_last=False)
    return tl, vl, tel

# ─────────────────────────────────────────────────────────────────────────────
# 8.  MODEL ARCHITECTURE
# ─────────────────────────────────────────────────────────────────────────────
class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 500, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        PE  = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        PE[:, 0::2] = torch.sin(pos * div)
        PE[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("PE", PE.unsqueeze(0))
    def forward(self, x):
        return self.dropout(x + self.PE[:, :x.size(1), :])

class SoilMoistureTransformer(nn.Module):
    def __init__(self, n_features, d_model, n_heads, n_layers, d_ff, horizon, dropout=0.1, seq_len=144):
        super().__init__()
        assert d_model % n_heads == 0
        self.input_proj = nn.Linear(n_features, d_model)
        self.pos_enc    = PositionalEncoding(d_model, dropout=dropout)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model = d_model, nhead = n_heads, dim_feedforward = d_ff,
            dropout = dropout, batch_first = True, norm_first = True, activation = "gelu"
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.norm    = nn.LayerNorm(d_model)

        # Flatten sequence input to fully protect spatial and temporal variances
        self.head = nn.Sequential(
            nn.Linear(seq_len * d_model, d_ff),    # 144 * 128 -> 512
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, horizon),              # 512 -> 1440
        )
        self._init_weights()

    def _init_weights(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def forward(self, x):
        x = self.input_proj(x)    # (batch, 144, 128)
        x = self.pos_enc(x)       # (batch, 144, 128)
        x = self.encoder(x)       # (batch, 144, 128)
        x = self.norm(x)          # (batch, 144, 128)

        x = x.reshape(x.size(0), -1)  # (batch, 144 * 128)
        return self.head(x)       # (batch, 1440)

def build_model(cfg):
    print("\n" + "=" * 65)
    print("STEP 6 — Build Transformer Model (Optimized Structural Flat Head)")
    print("=" * 65)
    model = SoilMoistureTransformer(
        n_features = N_FEATURES,
        d_model    = cfg["D_MODEL"],
        n_heads    = cfg["N_HEADS"],
        n_layers   = cfg["N_LAYERS"],
        d_ff       = cfg["D_FF"],
        horizon    = cfg["HORIZON"],
        dropout    = cfg["DROPOUT"],
        seq_len    = cfg["SEQ_LEN"]
    ).to(device)
    total = sum(p.numel() for p in model.parameters())
    dummy = torch.zeros(2, cfg["SEQ_LEN"], N_FEATURES).to(device)
    out   = model(dummy)
    print(f"  Input projection : ({N_FEATURES}) → ({cfg['D_MODEL']})")
    print(f"  Pos encoding     : sinusoidal, {cfg['SEQ_LEN']} positions")
    print(f"  Encoder layers   : {cfg['N_LAYERS']} × Pre-LN [MHA({cfg['N_HEADS']} heads)]")
    print(f"  Head Projection  : Flatten Structural ({cfg['SEQ_LEN']} * {cfg['D_MODEL']}) → {cfg['D_FF']} → {cfg['HORIZON']}")
    print(f"  Total parameters : {total:,}")
    print(f"  Shape check      : {tuple(dummy.shape)} → {tuple(out.shape)}   ✓ ")
    return model

# ─────────────────────────────────────────────────────────────────────────────
# 9.  TRAINING LOOP
# ─────────────────────────────────────────────────────────────────────────────
def train_model(model, train_dl, val_dl, cfg):
    print("\n" + "=" * 65)
    print("STEP 7 — Training")
    print("=" * 65)
    print(f"  Epochs     : {cfg['EPOCHS']}")
    print(f"  Batch size : {cfg['BATCH_SIZE']}")
    print(f"  LR         : {cfg['LR']} → 1e-6 (CosineAnnealing)")
    print(f"  Patience   : {cfg['PATIENCE']} epochs")
    print()
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg["LR"], weight_decay=cfg["WEIGHT_DECAY"])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg["EPOCHS"], eta_min=1e-6)
    criterion = nn.MSELoss()
    best_val   = float("inf")
    best_state = None
    patience   = 0
    t_losses, v_losses = [], []

    print(f"  {'Ep':<4}  {'Train MSE':<14}  {'Val MSE':<14}  {'LR':<9}  Status")
    print("-" * 65)
    for ep in range(1, cfg["EPOCHS"] + 1):
        model.train()
        t_loss = 0.0
        for bx, by in train_dl:
            bx, by = bx.to(device), by.to(device)
            optimizer.zero_grad()
            pred = model(bx)
            loss = criterion(pred, by)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            t_loss += loss.item() * len(bx)
        t_loss /= len(train_dl.dataset)
        t_losses.append(t_loss)

        model.eval()
        v_loss = 0.0
        with torch.no_grad():
            for bx, by in val_dl:
                bx, by = bx.to(device), by.to(device)
                v_loss += criterion(model(bx), by).item() * len(bx)
        v_loss /= len(val_dl.dataset)
        v_losses.append(v_loss)

        curr_lr = optimizer.param_groups[0]["lr"]
        scheduler.step()

        status = ""
        if v_loss < best_val:
            best_val = v_loss
            best_state = {k: v.cpu() for k, v in model.state_dict().items()}
            patience = 0
            status = "✓ best"
        else:
            patience += 1

        if ep == 1 or ep % 5 == 0 or status != "":
            print(f"  {ep:>3d}    {t_loss:<14.7f}  {v_loss:<14.7f}  {curr_lr:.2e}  {status}")

        if patience >= cfg["PATIENCE"]:
            print(f"\n  Early stopping targeted at epoch {ep}")
            break

    model.load_state_dict(best_state)
    torch.save(best_state, cfg["SAVE_MODEL"])
    print(f"\n  Model architecture saved -> {cfg['SAVE_MODEL']}")
    return t_losses, v_losses

# ─────────────────────────────────────────────────────────────────────────────
# 10. EVALUATION ON TEST SET
# ─────────────────────────────────────────────────────────────────────────────
def evaluate_model_with_advanced_metrics(model, test_dl, y_te_orig, sy, y_train_raw):
    """
    Runs inference on the test set and prints a detailed breakdown of
    MAE, MAPE, RMSE, and MASE both globally and per forecast day.
    """
    print("\n" + "=" * 65)
    print("STEP 5 — ADVANCED PERFORMANCE METRICS EVALUATION")
    print("=" * 65)

    model.eval()
    preds_scaled = []
    with torch.no_grad():
        for bx, _ in test_dl:
            preds_scaled.append(model(bx.to(device)).cpu().numpy())

    preds_scaled = np.concatenate(preds_scaled, axis=0)
    preds_orig = sy.inverse_transform(preds_scaled)

    # Calculate Global Metrics across the entire 10-day forecast matrix
    g_mae, g_mape, g_rmse, g_mase = calculate_advanced_metrics(y_te_orig, preds_orig, y_train_raw)
    g_r2 = r2_score(y_te_orig, preds_orig)

    print(f"\n  OVERALL GLOBAL METRICS (All 1440 sequence steps combined):")
    print(f"    RMSE : {g_rmse:.6f}")
    print(f"    MAE  : {g_mae:.6f}")
    print(f"    MAPE : {g_mape:.4f}%")
    print(f"    MASE : {g_mase:.4f}  (Values < 1 indicate the model beats a Naive 1-step baseline)")
    print(f"    R²   : {g_r2:.4f}")

    # Calculate Day-by-Day breakdown metrics (1 day = 144 steps of 10-mins)
    print(f"\n  DAY-BY-DAY TIMELINE EVALUATION BREAKDOWN:")
    print(f"   {'Day':<5} | {'RMSE':<10} | {'MAE':<10} | {'MAPE (%)':<10} | {'MASE':<10} | {'R²':<8}")
    print("  " + "-" * 62)

    for d in range(10):
        start, end = d * 144, (d + 1) * 144
        d_actual = y_te_orig[:, start:end]
        d_predicted = preds_orig[:, start:end]

        d_mae, d_mape, d_rmse, d_mase = calculate_advanced_metrics(d_actual, d_predicted, y_train_raw)
        d_r2 = r2_score(d_actual, d_predicted)

        print(f"   {d+1:>2d}   | {d_rmse:.6f} | {d_mae:.6f} | {d_mape:>7.4f}% | {d_mase:.4f} | {d_r2:.4f}")

    return preds_orig

# ─────────────────────────────────────────────────────────────────────────────
# 11. EXECUTION RUNNER
# ─────────────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    if not os.path.exists(CFG["DATA_PATH"]):
        print(f"Error: Target path '{CFG['DATA_PATH']}' not found. Verify location details.")
    else:
        raw_df = load_and_explore(CFG["DATA_PATH"])
        df_10 = resample_to_10min(raw_df, CFG["RESAMPLE_MIN"])
        df_feats = engineer_features(df_10)
        X, y = make_windows(df_feats, CFG["SEQ_LEN"], CFG["HORIZON"])

        X_tr_s, y_tr_s, X_v_s, y_v_s, X_te_s, y_te, y_te_s, sx, sy = split_and_scale(
            X, y, CFG["TRAIN_FRAC"], CFG["VAL_FRAC"]
        )

        train_dl, val_dl, test_dl = make_loaders(
            X_tr_s, y_tr_s, X_v_s, y_v_s, X_te_s, y_te_s, CFG["BATCH_SIZE"]
        )

        model = build_model(CFG)
        t_losses, v_losses = train_model(model, train_dl, val_dl, CFG)
        preds_orig = evaluate_model_with_advanced_metrics(model, test_dl, y_te, sy)

        print("\n" + "=" * 65)
        print("FINAL PIPELINE EXECUTION SUMMARY")
        print("=" * 65)
        print("Optimization successfully finished without length errors.")

Device : cuda

STEP 1 — Load & Explore Raw Data

STEP 2 — Creating Sliding Windows (Strict Timeline Structure)
  Total continuous windows created: 8,417

STEP 3 — Chronological Splitting & Scaling
  Train Block Windows : 5,891
  Val Block Windows   : 1,263
  Test Block Windows  : 1,263

Building Vanilla Transformer module...

STEP 4 — Training Vanilla Transformer Loop
  Ep    Train MSE       Val MSE         Status
-------------------------------------------------------
    1    1.0102117       1.0161408       ✓ best
    2    1.0004199       1.0159895       ✓ best
    3    1.0002954       1.0158843       ✓ best
    4    1.0002103       1.0158101       ✓ best
    5    1.0001501       1.0157674       ✓ best
    6    1.0001099       1.0157314       ✓ best
    7    1.0000841       1.0157089       ✓ best
    8    1.0000654       1.0156900       ✓ best
    9    1.0000545       1.0156767       ✓ best
   10    1.0000461       1.0156720       ✓ best
   11    1.0000413       1.0156647       ✓ bes

In [ ]:
"""
================================================================================
  SOIL MOISTURE 10-DAY AHEAD PREDICTION — PATCHTST (FIXED, FULL & ADVANCED)
================================================================================
"""
# ─────────────────────────────────────────────────────────────────────────────
# 0.  IMPORTS
# ─────────────────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings
import os

warnings.filterwarnings("ignore")
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {device}")

# ─────────────────────────────────────────────────────────────────────────────
# 1.  CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────
CFG = dict(
    DATA_PATH    = "/content/Final_Dataset.xlsx",
    RESAMPLE_MIN = 10,     # 10-min aggregation blocks
    SEQ_LEN      = 144,    # look-back: 144 blocks = 1 day
    HORIZON      = 1440,   # forecast: 1440 blocks = 10 days
    TRAIN_FRAC   = 0.70,
    VAL_FRAC     = 0.15,

    # ── PatchTST Architecture Hyperparameters ────────────────────────────────
    PATCH_LEN    = 16,     # Length of each overlapping sub-series patch
    STRIDE       = 8,      # Overlap stride between patches
    D_MODEL      = 128,    # Embedding space size
    N_HEADS      = 4,
    N_LAYERS     = 3,
    D_FF         = 256,
    DROPOUT      = 0.10,

    # ── Training parameters ──────────────────────────────────────────────────
    BATCH_SIZE   = 64,
    LR           = 4e-4,
    WEIGHT_DECAY = 1e-6,
    EPOCHS       = 100,
    PATIENCE     = 25,
    SAVE_MODEL   = "best_patchtst_sm_model.pt"
)

FEATURE_COLS = [
    "irrigation", "sm", "sm_roll6", "sm_roll24", "sm_diff1",
    "sm_lag1", "sm_lag6", "sm_lag24", "sin_time", "cos_time"
]
N_FEATURES = len(FEATURE_COLS)   # 10

# ─────────────────────────────────────────────────────────────────────────────
# 2.  DATA GENERATION PIPELINE (CHRONOLOGICAL & LEAK-FREE)
# ─────────────────────────────────────────────────────────────────────────────
def load_and_explore(path: str) -> pd.DataFrame:
    print("\n" + "=" * 65)
    print("STEP 1 — Load & Explore Raw Data")
    print("=" * 65)
    df = pd.read_excel(path)
    if len(df.columns) == 3:
        df = df.iloc[:, 1:]  # Automatically drop 'Minute' index if present
    df.columns = ["irrigation", "sm"]
    df["irrigation"] = df["irrigation"].ffill()
    return df

def resample_to_10min(df: pd.DataFrame, block: int = 10) -> pd.DataFrame:
    n_full = len(df) // block
    sm_arr  = df["sm"].values[:n_full * block].reshape(n_full, block).mean(axis=1)
    irr_arr = df["irrigation"].values[:n_full * block].reshape(n_full, block).mean(axis=1)
    return pd.DataFrame({"irrigation": irr_arr, "sm": sm_arr})

def engineer_features(df10: pd.DataFrame) -> pd.DataFrame:
    df = df10.copy()
    df["sm_roll6"]  = df["sm"].rolling(6,  min_periods=1).mean()
    df["sm_roll24"] = df["sm"].rolling(24, min_periods=1).mean()
    df["sm_diff1"]  = df["sm"].diff().fillna(0)
    df["sm_lag1"]   = df["sm"].shift(1).bfill()
    df["sm_lag6"]   = df["sm"].shift(6).bfill()
    df["sm_lag24"]  = df["sm"].shift(24).bfill()

    # Cyclical day tracking components
    df["block_of_day"] = df.index % 144
    df["sin_time"] = np.sin(2 * np.pi * df["block_of_day"] / 144)
    df["cos_time"] = np.cos(2 * np.pi * df["block_of_day"] / 144)
    df.drop(columns=["block_of_day"], inplace=True)
    return df.dropna().reset_index(drop=True)

def make_windows(df: pd.DataFrame, seq_len: int, horizon: int) -> tuple:
    print("\n" + "=" * 65)
    print("STEP 2 — Creating Sliding Windows (Strict Timeline Structure)")
    print("=" * 65)
    feat   = df[FEATURE_COLS].values.astype(np.float32)
    target = df["sm"].values.astype(np.float32)
    T      = len(feat)
    X_list, y_list = [], []
    for i in range(T - seq_len - horizon + 1):
        X_list.append(feat[i : i + seq_len])
        y_list.append(target[i + seq_len : i + seq_len + horizon])
    X = np.array(X_list, dtype=np.float32)
    y = np.array(y_list, dtype=np.float32)

    print(f"  Total continuous windows created: {len(X):,}")
    return X, y

def split_and_scale(X, y, train_frac, val_frac):
    print("\n" + "=" * 65)
    print("STEP 3 — Chronological Splitting & Scaling")
    print("=" * 65)
    N     = len(X)
    t_end = int(train_frac * N)
    v_end = int((train_frac + val_frac) * N)

    # Split linearly across temporal edges to preserve forecasting reality
    X_tr, y_tr = X[:t_end],      y[:t_end]
    X_v,  y_v  = X[t_end:v_end], y[t_end:v_end]
    X_te, y_te = X[v_end:],      y[v_end:]

    print(f"  Train Block Windows : {len(X_tr):,}")
    print(f"  Val Block Windows   : {len(X_v):,}")
    print(f"  Test Block Windows  : {len(X_te):,}")

    sx = StandardScaler()
    sy = StandardScaler()

    def sc_X(arr, fit=False):
        s = arr.shape
        flat = arr.reshape(-1, N_FEATURES)
        return (sx.fit_transform(flat) if fit else sx.transform(flat)).reshape(s)

    X_tr_s = sc_X(X_tr, fit=True)
    X_v_s  = sc_X(X_v)
    X_te_s = sc_X(X_te)
    y_tr_s = sy.fit_transform(y_tr)
    y_v_s  = sy.transform(y_v)
    y_te_s = sy.transform(y_te)

    return X_tr_s, y_tr_s, X_v_s, y_v_s, X_te_s, y_te, y_te_s, sy

# ─────────────────────────────────────────────────────────────────────────────
# 3.  UNIVERSAL ADVANCED EVALUATION ENGINE MODULE
# ─────────────────────────────────────────────────────────────────────────────
def calculate_advanced_metrics(y_true, y_pred, y_train_raw):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)

    # MAPE with safety epsilon boundary shift against zero division
    epsilon = 1e-8
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + epsilon))) * 100

    # MASE metric calculation relative to continuous unscaled training patterns
    naive_baseline_denom = np.mean(np.abs(np.diff(y_train_raw)))
    if naive_baseline_denom < 1e-8:
        naive_baseline_denom = 1e-8
    mase = mae / naive_baseline_denom

    return mae, mape, rmse, mase

def run_universal_evaluation(model, test_dl, y_te_orig, sy, y_train_raw):
    print("\n" + "=" * 65)
    print("UNIVERSAL TIME-SERIES EVALUATION (MAE, MAPE, RMSE, MASE)")
    print("=" * 65)

    model.eval()
    preds_scaled = []
    with torch.no_grad():
        for bx, _ in test_dl:
            preds_scaled.append(model(bx.to(device)).cpu().numpy())

    preds_scaled = np.concatenate(preds_scaled, axis=0)
    preds_orig = sy.inverse_transform(preds_scaled)

    # Global Summary
    g_mae, g_mape, g_rmse, g_mase = calculate_advanced_metrics(y_te_orig, preds_orig, y_train_raw)
    g_r2 = r2_score(y_te_orig, preds_orig)

    print(f"\n  OVERALL GLOBAL METRICS (All 1440 sequence steps combined):")
    print(f"    RMSE : {g_rmse:.6f}")
    print(f"    MAE  : {g_mae:.6f}")
    print(f"    MAPE : {g_mape:.4f}%")
    print(f"    MASE : {g_mase:.4f}")
    print(f"    R²   : {g_r2:.4f}")

    # Day by Day breakdown metrics table
    print(f"\n  DAY-BY-DAY TIMELINE EVALUATION BREAKDOWN:")
    print(f"   {'Day':<5} | {'RMSE':<10} | {'MAE':<10} | {'MAPE (%)':<10} | {'MASE':<10} | {'R²':<8}")
    print("  " + "-" * 62)

    for d in range(10):
        start, end = d * 144, (d + 1) * 144
        d_actual = y_te_orig[:, start:end]
        d_predicted = preds_orig[:, start:end]

        d_mae, d_mape, d_rmse, d_mase = calculate_advanced_metrics(d_actual, d_predicted, y_train_raw)
        d_r2 = r2_score(d_actual, d_predicted)

        print(f"   {d+1:>2d}   | {d_rmse:.6f} | {d_mae:.6f} | {d_mape:>7.4f}% | {d_mase:.4f} | {d_r2:.4f}")

    return preds_orig

# ─────────────────────────────────────────────────────────────────────────────
# 4.  PATCHTST MODEL ARCHITECTURE
# ─────────────────────────────────────────────────────────────────────────────
class PatchTSTEmbedding(nn.Module):
    def __init__(self, patch_len, d_model, dropout=0.1):
        super().__init__()
        self.patch_proj = nn.Linear(patch_len, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = self.patch_proj(x)
        return self.dropout(x)

class PatchTSTBackbone(nn.Module):
    def __init__(self, num_patches, patch_len, d_model, n_heads, n_layers, d_ff, dropout):
        super().__init__()
        self.embedding = PatchTSTEmbedding(patch_len, d_model, dropout)
        self.pos_emb = nn.Parameter(torch.zeros(1, num_patches, d_model))

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_ff,
            dropout=dropout, batch_first=True, norm_first=True, activation="gelu"
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)

    def forward(self, x):
        x = self.embedding(x)
        x = x + self.pos_emb
        return self.transformer(x)

class PatchTST(nn.Module):
    def __init__(self, seq_len, horizon, n_features, patch_len, stride, d_model, n_heads, n_layers, d_ff, dropout):
        super().__init__()
        self.seq_len = seq_len
        self.horizon = horizon
        self.n_features = n_features
        self.patch_len = patch_len
        self.stride = stride

        assert (seq_len - patch_len) % stride == 0, "SEQ_LEN - PATCH_LEN must be divisible by STRIDE."
        self.num_patches = int((seq_len - patch_len) / stride) + 1

        self.backbone = PatchTSTBackbone(
            self.num_patches, patch_len, d_model, n_heads, n_layers, d_ff, dropout
        )

        self.head_in_features = self.num_patches * d_model
        self.head = nn.Sequential(
            nn.Linear(self.head_in_features, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, horizon)
        )

    def forward(self, x):
        batch_size = x.size(0)

        # Enforce Channel Independence: process columns independently inside the Transformer batch axis
        x = x.transpose(1, 2)
        x = x.reshape(batch_size * self.n_features, self.seq_len)

        # Slicing temporal spaces into patches via unfold
        x = x.unfold(dimension=-1, size=self.patch_len, step=self.stride)

        enc_out = self.backbone(x)
        enc_out = enc_out.reshape(enc_out.size(0), -1)

        preds = self.head(enc_out)
        preds = preds.reshape(batch_size, self.n_features, self.horizon)

        # Isolate target output strictly corresponding to index [1] ('sm')
        return preds[:, 1, :]

# ─────────────────────────────────────────────────────────────────────────────
# 5.  TRAINING LOOP ENGINE
# ─────────────────────────────────────────────────────────────────────────────
def train_patchtst(model, train_dl, val_dl, cfg):
    print("\n" + "=" * 65)
    print("STEP 4 — Training PatchTST Loops")
    print("=" * 65)
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg["LR"], weight_decay=cfg["WEIGHT_DECAY"])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg["EPOCHS"], eta_min=1e-6)
    criterion = nn.MSELoss()

    best_val = float("inf")
    best_state = None
    patience = 0

    print(f"  {'Ep':<4}  {'Train MSE':<14}  {'Val MSE':<14}  Status")
    print("-" * 55)
    for ep in range(1, cfg["EPOCHS"] + 1):
        model.train()
        t_loss = 0.0
        for bx, by in train_dl:
            bx, by = bx.to(device), by.to(device)
            optimizer.zero_grad()
            loss = criterion(model(bx), by)
            loss.backward()
            optimizer.step()
            t_loss += loss.item() * len(bx)
        t_loss /= len(train_dl.dataset)

        model.eval()
        v_loss = 0.0
        with torch.no_grad():
            for bx, by in val_dl:
                bx, by = bx.to(device), by.to(device)
                v_loss += criterion(model(bx), by).item() * len(bx)
        v_loss /= len(val_dl.dataset)

        scheduler.step()
        status = ""
        if v_loss < best_val:
            best_val = v_loss
            best_state = {k: v.cpu() for k, v in model.state_dict().items()}
            patience = 0
            status = "✓ best"
        else:
            patience += 1

        if ep == 1 or ep % 5 == 0 or status != "":
            print(f"  {ep:>3d}    {t_loss:<14.7f}  {v_loss:<14.7f}  {status}")

        if patience >= cfg["PATIENCE"]:
            print(f"\n  Early stopping targeted at epoch {ep}")
            break

    model.load_state_dict(best_state)
    torch.save(best_state, cfg["SAVE_MODEL"])
    return model

# ─────────────────────────────────────────────────────────────────────────────
# 6.  MAIN EXECUTIVE PIPELINE RUNNER
# ─────────────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    if not os.path.exists(CFG["DATA_PATH"]):
        print(f"File path reference error. Verify target: {CFG['DATA_PATH']}")
    else:
        raw_df = load_and_explore(CFG["DATA_PATH"])
        df_10 = resample_to_10min(raw_df, CFG["RESAMPLE_MIN"])
        df_feats = engineer_features(df_10)
        X, y = make_windows(df_feats, CFG["SEQ_LEN"], CFG["HORIZON"])

        X_tr_s, y_tr_s, X_v_s, y_v_s, X_te_s, y_te, y_te_s, sy = split_and_scale(
            X, y, CFG["TRAIN_FRAC"], CFG["VAL_FRAC"]
        )

        # Capture unscaled continuous raw train target blocks to pass into the MASE denominator
        N_train_windows = len(y_tr_s)
        y_train_raw_sequence = y[:N_train_windows].flatten()

        # DataLoader configurations: Shuffling is activated only on training sequences
        train_loader = DataLoader(TensorDataset(torch.tensor(X_tr_s), torch.tensor(y_tr_s)), CFG["BATCH_SIZE"], shuffle=True)
        val_loader   = DataLoader(TensorDataset(torch.tensor(X_v_s),  torch.tensor(y_v_s)),  CFG["BATCH_SIZE"], shuffle=False)
        test_loader  = DataLoader(TensorDataset(torch.tensor(X_te_s), torch.tensor(y_te_s)), CFG["BATCH_SIZE"], shuffle=False)

        print("\nBuilding PatchTST Architecture module...")
        model = PatchTST(
            seq_len=CFG["SEQ_LEN"], horizon=CFG["HORIZON"], n_features=N_FEATURES,
            patch_len=CFG["PATCH_LEN"], stride=CFG["STRIDE"], d_model=CFG["D_MODEL"],
            n_heads=CFG["N_HEADS"], n_layers=CFG["N_LAYERS"], d_ff=CFG["D_FF"], dropout=CFG["DROPOUT"]
        ).to(device)

        print(f"  Total feature paths embedded    : {N_FEATURES}")
        print(f"  Calculated sub-series patches   : {model.num_patches} sequential tokens")

        # Run execution sequences
        model = train_patchtst(model, train_loader, val_loader, CFG)
        run_universal_evaluation(model, test_loader, y_te, sy, y_train_raw_sequence)

Device : cuda

STEP 1 — Load & Explore Raw Data

STEP 2 — Creating Sliding Windows (Strict Timeline Structure)
  Total continuous windows created: 8,417

STEP 3 — Chronological Splitting & Scaling
  Train Block Windows : 5,891
  Val Block Windows   : 1,263
  Test Block Windows  : 1,263

Building PatchTST Architecture module...
  Total feature paths embedded    : 10
  Calculated sub-series patches   : 17 sequential tokens

STEP 4 — Training PatchTST Loops
  Ep    Train MSE       Val MSE         Status
-------------------------------------------------------
    1    1.0024513       1.0163872       ✓ best
    2    1.0006116       1.0160789       ✓ best
    3    1.0003705       1.0158968       ✓ best
    4    1.0002319       1.0157963       ✓ best
    5    1.0001512       1.0157377       ✓ best
    6    1.0001033       1.0157014       ✓ best
    7    1.0000773       1.0156803       ✓ best
    8    1.0000612       1.0156714       ✓ best
    9    1.0000519       1.0156601       ✓ best
   10 

In [ ]:
"""
================================================================================
  SOIL MOISTURE 10-DAY AHEAD PREDICTION — iTRANSFORMER (FIXED, FULL & ADVANCED)
================================================================================
"""
# ─────────────────────────────────────────────────────────────────────────────
# 0.  IMPORTS
# ─────────────────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings
import os

warnings.filterwarnings("ignore")
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {device}")

# ─────────────────────────────────────────────────────────────────────────────
# 1.  CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────
CFG = dict(
    DATA_PATH    = "/content/Final_Dataset.xlsx",
    RESAMPLE_MIN = 10,     # 10-min aggregation blocks
    SEQ_LEN      = 144,    # look-back: 144 blocks = 1 day
    HORIZON      = 1440,   # forecast: 1440 blocks = 10 days
    TRAIN_FRAC   = 0.70,
    VAL_FRAC     = 0.15,

    # ── iTransformer Architecture Hyperparameters ───────────────────────────
    D_MODEL      = 128,    # Dimension to project the entire SEQ_LEN timeline into
    N_HEADS      = 4,
    N_LAYERS     = 3,
    D_FF         = 256,
    DROPOUT      = 0.10,

    # ── Training parameters ──────────────────────────────────────────────────
    BATCH_SIZE   = 64,
    LR           = 5e-4,
    WEIGHT_DECAY = 1e-6,
    EPOCHS       = 100,
    PATIENCE     = 25,
    SAVE_MODEL   = "best_itransformer_sm_model.pt"
)

FEATURE_COLS = [
    "irrigation", "sm", "sm_roll6", "sm_roll24", "sm_diff1",
    "sm_lag1", "sm_lag6", "sm_lag24", "sin_time", "cos_time"
]
N_FEATURES = len(FEATURE_COLS)   # 10 inverted tokens populate the attention layers

# ─────────────────────────────────────────────────────────────────────────────
# 2.  DATA GENERATION PIPELINE (CHRONOLOGICAL & LEAK-FREE)
# ─────────────────────────────────────────────────────────────────────────────
def load_and_explore(path: str) -> pd.DataFrame:
    print("\n" + "=" * 65)
    print("STEP 1 — Load & Explore Raw Data")
    print("=" * 65)
    df = pd.read_excel(path)
    if len(df.columns) == 3:
        df = df.iloc[:, 1:]  # Automatically drop 'Minute' index if present
    df.columns = ["irrigation", "sm"]
    df["irrigation"] = df["irrigation"].ffill()
    return df

def resample_to_10min(df: pd.DataFrame, block: int = 10) -> pd.DataFrame:
    n_full = len(df) // block
    sm_arr  = df["sm"].values[:n_full * block].reshape(n_full, block).mean(axis=1)
    irr_arr = df["irrigation"].values[:n_full * block].reshape(n_full, block).mean(axis=1)
    return pd.DataFrame({"irrigation": irr_arr, "sm": sm_arr})

def engineer_features(df10: pd.DataFrame) -> pd.DataFrame:
    df = df10.copy()
    df["sm_roll6"]  = df["sm"].rolling(6,  min_periods=1).mean()
    df["sm_roll24"] = df["sm"].rolling(24, min_periods=1).mean()
    df["sm_diff1"]  = df["sm"].diff().fillna(0)
    df["sm_lag1"]   = df["sm"].shift(1).bfill()
    df["sm_lag6"]   = df["sm"].shift(6).bfill()
    df["sm_lag24"]  = df["sm"].shift(24).bfill()

    # Cyclical day tracking components
    df["block_of_day"] = df.index % 144
    df["sin_time"] = np.sin(2 * np.pi * df["block_of_day"] / 144)
    df["cos_time"] = np.cos(2 * np.pi * df["block_of_day"] / 144)
    df.drop(columns=["block_of_day"], inplace=True)
    return df.dropna().reset_index(drop=True)

def make_windows(df: pd.DataFrame, seq_len: int, horizon: int) -> tuple:
    print("\n" + "=" * 65)
    print("STEP 2 — Creating Sliding Windows (Strict Timeline Structure)")
    print("=" * 65)
    feat   = df[FEATURE_COLS].values.astype(np.float32)
    target = df["sm"].values.astype(np.float32)
    T      = len(feat)
    X_list, y_list = [], []
    for i in range(T - seq_len - horizon + 1):
        X_list.append(feat[i : i + seq_len])
        y_list.append(target[i + seq_len : i + seq_len + horizon])
    X = np.array(X_list, dtype=np.float32)
    y = np.array(y_list, dtype=np.float32)

    print(f"  Total continuous windows created: {len(X):,}")
    return X, y

def split_and_scale(X, y, train_frac, val_frac):
    print("\n" + "=" * 65)
    print("STEP 3 — Chronological Splitting & Scaling")
    print("=" * 65)
    N     = len(X)
    t_end = int(train_frac * N)
    v_end = int((train_frac + val_frac) * N)

    # Split linearly across chronological chunks to secure an honest holdout test environment
    X_tr, y_tr = X[:t_end],      y[:t_end]
    X_v,  y_v  = X[t_end:v_end], y[t_end:v_end]
    X_te, y_te = X[v_end:],      y[v_end:]

    print(f"  Train Block Windows : {len(X_tr):,}")
    print(f"  Val Block Windows   : {len(X_v):,}")
    print(f"  Test Block Windows  : {len(X_te):,}")

    sx = StandardScaler()
    sy = StandardScaler()

    def sc_X(arr, fit=False):
        s = arr.shape
        flat = arr.reshape(-1, N_FEATURES)
        return (sx.fit_transform(flat) if fit else sx.transform(flat)).reshape(s)

    X_tr_s = sc_X(X_tr, fit=True)
    X_v_s  = sc_X(X_v)
    X_te_s = sc_X(X_te)
    y_tr_s = sy.fit_transform(y_tr)
    y_v_s  = sy.transform(y_v)
    y_te_s = sy.transform(y_te)

    return X_tr_s, y_tr_s, X_v_s, y_v_s, X_te_s, y_te, y_te_s, sy

# ─────────────────────────────────────────────────────────────────────────────
# 3.  UNIVERSAL ADVANCED EVALUATION ENGINE MODULE
# ─────────────────────────────────────────────────────────────────────────────
def calculate_advanced_metrics(y_true, y_pred, y_train_raw):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)

    # MAPE with protective epsilon shift preventing zero division errors
    epsilon = 1e-8
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + epsilon))) * 100

    # MASE calculation relative to historical continuous train patterns
    naive_baseline_denom = np.mean(np.abs(np.diff(y_train_raw)))
    if naive_baseline_denom < 1e-8:
        naive_baseline_denom = 1e-8
    mase = mae / naive_baseline_denom

    return mae, mape, rmse, mase

def run_universal_evaluation(model, test_dl, y_te_orig, sy, y_train_raw):
    print("\n" + "=" * 65)
    print("UNIVERSAL TIME-SERIES EVALUATION (MAE, MAPE, RMSE, MASE)")
    print("=" * 65)

    model.eval()
    preds_scaled = []
    with torch.no_grad():
        for bx, _ in test_dl:
            preds_scaled.append(model(bx.to(device)).cpu().numpy())

    preds_scaled = np.concatenate(preds_scaled, axis=0)
    preds_orig = sy.inverse_transform(preds_scaled)

    # Global Overview Summary
    g_mae, g_mape, g_rmse, g_mase = calculate_advanced_metrics(y_te_orig, preds_orig, y_train_raw)
    g_r2 = r2_score(y_te_orig, preds_orig)

    print(f"\n  OVERALL GLOBAL METRICS (All 1440 sequence steps combined):")
    print(f"    RMSE : {g_rmse:.6f}")
    print(f"    MAE  : {g_mae:.6f}")
    print(f"    MAPE : {g_mape:.4f}%")
    print(f"    MASE : {g_mase:.4f}")
    print(f"    R²   : {g_r2:.4f}")

    # Day by Day breakdown metrics table
    print(f"\n  DAY-BY-DAY TIMELINE EVALUATION BREAKDOWN:")
    print(f"   {'Day':<5} | {'RMSE':<10} | {'MAE':<10} | {'MAPE (%)':<10} | {'MASE':<10} | {'R²':<8}")
    print("  " + "-" * 62)

    for d in range(10):
        start, end = d * 144, (d + 1) * 144
        d_actual = y_te_orig[:, start:end]
        d_predicted = preds_orig[:, start:end]

        d_mae, d_mape, d_rmse, d_mase = calculate_advanced_metrics(d_actual, d_predicted, y_train_raw)
        d_r2 = r2_score(d_actual, d_predicted)

        print(f"   {d+1:>2d}   | {d_rmse:.6f} | {d_mae:.6f} | {d_mape:>7.4f}% | {d_mase:.4f} | {d_r2:.4f}")

    return preds_orig

# ─────────────────────────────────────────────────────────────────────────────
# 4.  iTRANSFORMER MODEL ARCHITECTURE
# ─────────────────────────────────────────────────────────────────────────────
class iTransformer(nn.Module):
    """
    The Inverted Transformer (iTransformer) Module.
    Embeds the entire temporal timeline of individual channels as sequence tokens.
    """
    def __init__(self, seq_len, horizon, n_features, d_model, n_heads, n_layers, d_ff, dropout):
        super().__init__()
        self.seq_len = seq_len
        self.horizon = horizon
        self.n_features = n_features

        # Temporal Linear Embedding Projection
        self.enc_embedding = nn.Linear(seq_len, d_model)
        self.norm = nn.LayerNorm(seq_len)

        # Attention handles interactions across multivariate columns (features)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_ff,
            dropout=dropout, batch_first=True, norm_first=True, activation="gelu"
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)

        # Projection output mapper
        self.head = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, horizon)
        )

    def forward(self, x):
        # Input shape matrix layout: (batch_size, seq_len, n_features)
        batch_size = x.size(0)

        # Transpose to invert dimensions -> (batch_size, n_features, seq_len)
        x = x.transpose(1, 2)

        # Normalize and map individual raw continuous timeline features out to d_model space
        x = self.norm(x)
        x = self.enc_embedding(x) # (batch_size, n_features, d_model)

        # Run global self-attention across tokens (channels)
        enc_out = self.encoder(x) # (batch_size, n_features, d_model)

        # Run forecasting heads
        preds = self.head(enc_out) # (batch_size, n_features, horizon)

        # Isolate soil moisture predictions index [1] ('sm')
        return preds[:, 1, :]

# ─────────────────────────────────────────────────────────────────────────────
# 5.  TRAINING LOOP ENGINE
# ─────────────────────────────────────────────────────────────────────────────
def train_itransformer(model, train_dl, val_dl, cfg):
    print("\n" + "=" * 65)
    print("STEP 4 — Training iTransformer Loops")
    print("=" * 65)
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg["LR"], weight_decay=cfg["WEIGHT_DECAY"])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg["EPOCHS"], eta_min=1e-6)
    criterion = nn.MSELoss()

    best_val = float("inf")
    best_state = None
    patience = 0

    print(f"  {'Ep':<4}  {'Train MSE':<14}  {'Val MSE':<14}  Status")
    print("-" * 55)
    for ep in range(1, cfg["EPOCHS"] + 1):
        model.train()
        t_loss = 0.0
        for bx, by in train_dl:
            bx, by = bx.to(device), by.to(device)
            optimizer.zero_grad()
            loss = criterion(model(bx), by)
            loss.backward()
            optimizer.step()
            t_loss += loss.item() * len(bx)
        t_loss /= len(train_dl.dataset)

        model.eval()
        v_loss = 0.0
        with torch.no_grad():
            for bx, by in val_dl:
                bx, by = bx.to(device), by.to(device)
                v_loss += criterion(model(bx), by).item() * len(bx)
        v_loss /= len(val_dl.dataset)

        scheduler.step()
        status = ""
        if v_loss < best_val:
            best_val = v_loss
            best_state = {k: v.cpu() for k, v in model.state_dict().items()}
            patience = 0
            status = "✓ best"
        else:
            patience += 1

        if ep == 1 or ep % 5 == 0 or status != "":
            print(f"  {ep:>3d}    {t_loss:<14.7f}  {v_loss:<14.7f}  {status}")

        if patience >= cfg["PATIENCE"]:
            print(f"\n  Early stopping targeted at epoch {ep}")
            break

    model.load_state_dict(best_state)
    torch.save(best_state, cfg["SAVE_MODEL"])
    return model

# ─────────────────────────────────────────────────────────────────────────────
# 6.  MAIN EXECUTIVE PIPELINE RUNNER
# ─────────────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    if not os.path.exists(CFG["DATA_PATH"]):
        print(f"File path reference error. Verify target: {CFG['DATA_PATH']}")
    else:
        raw_df = load_and_explore(CFG["DATA_PATH"])
        df_10 = resample_to_10min(raw_df, CFG["RESAMPLE_MIN"])
        df_feats = engineer_features(df_10)
        X, y = make_windows(df_feats, CFG["SEQ_LEN"], CFG["HORIZON"])

        X_tr_s, y_tr_s, X_v_s, y_v_s, X_te_s, y_te, y_te_s, sy = split_and_scale(
            X, y, CFG["TRAIN_FRAC"], CFG["VAL_FRAC"]
        )

        # Flatten raw continuous train sequence variables to formulate the MASE base denominator
        N_train_windows = len(y_tr_s)
        y_train_raw_sequence = y[:N_train_windows].flatten()

        # DataLoader assignments: Training batches shuffle cleanly while testing stays sequential
        train_loader = DataLoader(TensorDataset(torch.tensor(X_tr_s), torch.tensor(y_tr_s)), CFG["BATCH_SIZE"], shuffle=True)
        val_loader   = DataLoader(TensorDataset(torch.tensor(X_v_s),  torch.tensor(y_v_s)),  CFG["BATCH_SIZE"], shuffle=False)
        test_loader  = DataLoader(TensorDataset(torch.tensor(X_te_s), torch.tensor(y_te_s)), CFG["BATCH_SIZE"], shuffle=False)

        print("\nBuilding iTransformer Architecture module...")
        model = iTransformer(
            seq_len=CFG["SEQ_LEN"], horizon=CFG["HORIZON"], n_features=N_FEATURES,
            d_model=CFG["D_MODEL"], n_heads=CFG["N_HEADS"], n_layers=CFG["N_LAYERS"],
            d_ff=CFG["D_FF"], dropout=CFG["DROPOUT"]
        ).to(device)

        print(f"  Total Inverted Tokens (Channels) : {N_FEATURES}")
        print(f"  Temporal Vector Feature Dimension : {CFG['SEQ_LEN']} steps → Projected into {CFG['D_MODEL']}")

        # Run loops
        model = train_itransformer(model, train_loader, val_loader, CFG)
        run_universal_evaluation(model, test_loader, y_te, sy, y_train_raw_sequence)

Device : cuda

STEP 1 — Load & Explore Raw Data

STEP 2 — Creating Sliding Windows (Strict Timeline Structure)
  Total continuous windows created: 8,417

STEP 3 — Chronological Splitting & Scaling
  Train Block Windows : 5,891
  Val Block Windows   : 1,263
  Test Block Windows  : 1,263

Building iTransformer Architecture module...
  Total Inverted Tokens (Channels) : 10
  Temporal Vector Feature Dimension : 144 steps → Projected into 128

STEP 4 — Training iTransformer Loops
  Ep    Train MSE       Val MSE         Status
-------------------------------------------------------
    1    1.0057347       1.0162318       ✓ best
    2    1.0006505       1.0159317       ✓ best
    3    1.0003505       1.0158603       ✓ best
    4    1.0002154       1.0157555       ✓ best
    5    1.0001353       1.0157220       ✓ best
    6    1.0000531       1.0156927       ✓ best
   10    0.9994984       1.0157924       
   15    0.9967057       1.0168222       
   20    0.9915957       1.0185858       
   